In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
file_path = "D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/sub-01_ses-01_eeg-epo.fif"
epochs = mne.read_epochs(file_path, preload=True)
print(epochs)
print("Event IDs:", epochs.event_id)

In [ ]:
# 31=up, 32=down, 33=right, 34=left
label_map = {31: 'up', 32: 'down', 33: 'right', 34: 'left'}
classes = sorted(label_map.keys())
sfreq = epochs.info['sfreq']
ch_names = epochs.ch_names
print(f"Sampling rate: {sfreq} Hz, Channels: {len(ch_names)}")

In [ ]:
# Compute PSD (Welch) per class
psd_per_class = {}
for code in classes:
    label = label_map[code]
    ep = epochs[str(code)] if str(code) in epochs.event_id else epochs[label]
    spectrum = ep.compute_psd(method='welch', fmin=0.5, fmax=60.0, n_fft=256, verbose=False)
    psd_per_class[label] = spectrum.get_data()  # (n_epochs, n_channels, n_freqs)

freqs = spectrum.freqs
print(f"Frequency resolution: {freqs[1]-freqs[0]:.2f} Hz, bins: {len(freqs)}")

In [ ]:
# Mean PSD across epochs and channels per class
fig, ax = plt.subplots(figsize=(10, 5))
for label, psd in psd_per_class.items():
    mean_psd = psd.mean(axis=(0, 1))
    ax.semilogy(freqs, mean_psd, label=label)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density (V²/Hz)')
ax.set_title('Mean PSD per Inner-Speech Direction (sub-01, ses-01)')
ax.legend()
ax.set_xlim([0.5, 60])
plt.tight_layout()
plt.show()

In [ ]:
# Band power summary: delta, theta, alpha, beta, gamma
bands = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 60)}

print(f"{'Band':<8}", end='')
for label in [label_map[c] for c in classes]:
    print(f"{label:>10}", end='')
print()

for band_name, (flo, fhi) in bands.items():
    mask = (freqs >= flo) & (freqs < fhi)
    print(f"{band_name:<8}", end='')
    for label in [label_map[c] for c in classes]:
        bp = psd_per_class[label][:, :, mask].mean()
        print(f"{bp:>10.3e}", end='')
    print()